# 第四章：LLM 作为变异算子

**系列：** OpenEvolve — 从零到专家·手撕全流程  
**上一章：** [03 — 岛屿进化](./03-island-evolution.ipynb)  
**本章内容：** 为什么随机变异不够用、LLM 如何充当智能变异算子、Prompt 工程的核心模式、SEARCH/REPLACE 差异格式、以及 LLM 集成的从零实现。

---

## 随机变异的致命缺陷

回顾第一章，我们的变异器是这样的：随机选一行，随机修改几个字符。运行 200 次迭代后，分数几乎没有提升。

**为什么？** 因为代码不是随机字符串——它有语法、有语义、有逻辑结构。随机改一个字符，99% 的情况下会：
- 破坏语法（`for` 变成 `fxr`）
- 破坏逻辑（`<` 变成 `=`）
- 产生无意义的修改（注释里改了个字母）

这就像让一个不识字的人修改论文——他可以随机换字母，但改出好论文的概率几乎为零。

### LLM 的优势

LLM **理解代码**。它知道：
- `for i in range(n)` 是一个循环
- 把冒泡排序改成快速排序需要改哪些部分
- 哪些修改会提升性能，哪些会破坏正确性

**OpenEvolve 的核心洞察：用 LLM 替代随机变异，让每次变异都是「有意义的代码修改」。**

## 架构：Prompt → LLM → Diff

```mermaid
flowchart LR
    A["父代程序"] --> B["Prompt 构建器"]
    C["进化历史"] --> B
    D["灵感程序"] --> B
    B --> E["LLM 集成"]
    E --> F{"解析响应"}
    F -->|"差异模式"| G["SEARCH/REPLACE"]
    F -->|"重写模式"| H["完整新代码"]
    G --> I["应用差异"]
    H --> I
    I --> J["子代程序"]
```

三个核心组件：
1. **Prompt 构建器** — 把父代代码、进化历史、灵感程序组装成提示词
2. **LLM 集成** — 加权随机选择模型，调用 API 生成响应
3. **响应解析器** — 从 LLM 输出中提取代码修改，应用到父代上

## Prompt 工程：给 LLM 足够的上下文

一个好的变异 prompt 需要回答四个问题：

| 问题 | 对应的 Prompt 部分 | OpenEvolve 源码 |
|------|-------------------|----------------|
| 你是谁？ | System Message（专家开发者角色） | `prompts/defaults/system_message.txt` |
| 当前代码是什么？ | 父代程序的完整代码 | `prompt/sampler.py:build_prompt()` |
| 之前试过什么？ | 最近 3 次尝试的修改和结果 | `prompts/defaults/previous_attempt.txt` |
| 有什么灵感？ | 其他高分程序的代码片段 | `prompts/defaults/inspiration_program.txt` |

### 为什么需要进化历史？

没有历史信息，LLM 会反复提出**相同的修改建议**。有了历史，它知道：
- 「上次把循环展开了，分数提升了 5%」→ 继续这个方向
- 「上次尝试用递归，分数下降了」→ 避免这个方向

### 为什么需要灵感程序？

灵感程序来自其他高分程序（可能在不同岛屿上）。它们给 LLM 提供**跨家族的创意**——比如冒泡排序家族的 LLM 看到归并排序的代码后，可能会提出混合方案。

> **对应源码：** `openevolve/prompt/sampler.py` — `build_prompt()` 方法构建完整的 prompt

## SEARCH/REPLACE 差异格式

OpenEvolve 不要求 LLM 重写整个程序——它只需要指出**要修改的部分**。格式如下：

```
<<<<<<< SEARCH
def bubble_sort(arr):
    for i in range(len(arr)):
        for j in range(len(arr) - 1):
=======
def bubble_sort(arr):
    for i in range(len(arr)):
        for j in range(len(arr) - 1 - i):  # 优化：已排序部分不再比较
>>>>>>> REPLACE
```

### 为什么用差异而不是完整重写？

| | 差异模式 | 完整重写 |
|--|---------|--------|
| Token 用量 | 少（只输出改动部分） | 多（输出整个程序） |
| 精确度 | 高（明确指出改什么） | 低（可能意外改变其他部分） |
| 可审计性 | 好（一眼看出改了什么） | 差（需要 diff 工具对比） |
| 适用场景 | 大多数情况 | 需要大幅重构时 |

> **对应源码：** `openevolve/utils/code_utils.py` — `extract_diffs()` 和 `apply_diff()`

## 实现：从零构建 LLM 变异系统

我们将构建四个组件：
1. **Diff 解析器** — 从 LLM 响应中提取 SEARCH/REPLACE 块
2. **Diff 应用器** — 将修改应用到源代码
3. **Prompt 构建器** — 组装变异提示词
4. **LLM 集成接口** — 统一的模型调用接口（含模拟模式）

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import re
import random
from typing import Optional, List, Tuple
from dataclasses import dataclass, field

from our_implementation.core import Program

print("导入完成")

In [ ]:
def extract_diffs(text: str) -> list[tuple[str, str]]:
    """从 LLM 响应中提取 SEARCH/REPLACE 块。
    
    对应 OpenEvolve: utils/code_utils.py:extract_diffs()
    
    格式：
    <<<<<<< SEARCH
    要查找的原始代码
    =======
    替换成的新代码
    >>>>>>> REPLACE
    
    Returns:
        [(search_text, replace_text), ...] 的列表
    """
    pattern = r"<<<<<<< SEARCH\n(.*?)=======\n(.*?)>>>>>>> REPLACE"
    matches = re.findall(pattern, text, re.DOTALL)
    return [(s.strip('\n'), r.strip('\n')) for s, r in matches]


# 测试
test_response = '''我建议做以下优化：

<<<<<<< SEARCH
    for j in range(len(arr) - 1):
=======
    for j in range(len(arr) - 1 - i):
>>>>>>> REPLACE

这样已排序的部分就不会再被比较了。
'''

diffs = extract_diffs(test_response)
print(f"提取到 {len(diffs)} 个差异块：")
for i, (search, replace) in enumerate(diffs):
    print(f"  块 {i}: '{search}' -> '{replace}'")

In [ ]:
def apply_diff(original_code: str, diff_text: str) -> str:
    """将 SEARCH/REPLACE 差异应用到原始代码。
    
    对应 OpenEvolve: utils/code_utils.py:apply_diff()
    
    逐行匹配 SEARCH 块的内容，找到后替换为 REPLACE 块。
    多个 SEARCH/REPLACE 块按顺序应用。
    
    设计决策：
    - 使用逐行匹配而非简单字符串替换，更鲁棒
    - 如果 SEARCH 块找不到匹配，跳过并打印警告
    - 只替换第一次出现（避免意外修改）
    """
    diffs = extract_diffs(diff_text)
    result = original_code
    
    for search, replace in diffs:
        if search in result:
            # 直接字符串替换（只替换第一次出现）
            result = result.replace(search, replace, 1)
        else:
            # 尝试逐行匹配（忽略前后空白差异）
            search_lines = search.strip().splitlines()
            result_lines = result.splitlines()
            matched = False
            
            for i in range(len(result_lines) - len(search_lines) + 1):
                window = result_lines[i:i + len(search_lines)]
                if all(a.strip() == b.strip() for a, b in zip(window, search_lines)):
                    # 匹配成功，替换
                    replace_lines = replace.splitlines()
                    result_lines[i:i + len(search_lines)] = replace_lines
                    result = '\n'.join(result_lines)
                    matched = True
                    break
            
            if not matched:
                print(f"  [警告] SEARCH 块未匹配: '{search[:50]}...'")
    
    return result


# 测试
original = '''def bubble_sort(arr):
    for i in range(len(arr)):
        for j in range(len(arr) - 1):
            if arr[j] > arr[j+1]:
                arr[j], arr[j+1] = arr[j+1], arr[j]
    return arr'''

modified = apply_diff(original, test_response)
print("修改后的代码：")
print(modified)
print("\n注意第 3 行的变化：range(len(arr) - 1) -> range(len(arr) - 1 - i)")

In [ ]:
class PromptBuilder:
    """构建 LLM 变异提示词。
    
    对应 OpenEvolve: prompt/sampler.py:PromptSampler
    
    提示词结构：
    1. 系统消息 — 角色设定
    2. 当前代码 — 父代程序
    3. 进化历史 — 之前的尝试和结果
    4. 灵感程序 — 其他高分程序
    5. 指令 — 要求使用 SEARCH/REPLACE 格式
    """
    
    SYSTEM_MESSAGE = """你是一位专家级软件工程师。你的任务是改进给定的程序，
使其获得更高的评估分数。你应该：
1. 分析当前代码的不足
2. 参考进化历史，避免重复失败的尝试
3. 从灵感程序中汲取有价值的思路
4. 提出具体的代码修改

使用 SEARCH/REPLACE 格式输出你的修改。"""
    
    DIFF_INSTRUCTION = """请使用以下格式输出代码修改：

<<<<<<< SEARCH
要替换的原始代码（必须精确匹配）
=======
替换后的新代码
>>>>>>> REPLACE

你可以输出多个 SEARCH/REPLACE 块来做多处修改。"""
    
    def build_prompt(
        self,
        parent: Program,
        history: list[dict] = None,
        inspirations: list[Program] = None,
    ) -> tuple[str, str]:
        """构建系统消息和用户消息。
        
        Returns:
            (system_message, user_message)
        """
        parts = []
        
        # 1. 当前代码
        score = parent.metrics.get('score', 0.0)
        parts.append(f"## 当前程序（适应度: {score:.4f}）\n")
        parts.append(f"```python\n{parent.code}\n```\n")
        
        # 2. 进化历史
        if history:
            parts.append("## 最近的尝试\n")
            for h in history[-3:]:  # 最多显示 3 条
                outcome = "提升" if h.get('improved', False) else "下降"
                parts.append(
                    f"- 修改: {h.get('description', '未知')} "
                    f"-> 结果: {outcome} "
                    f"(分数: {h.get('score', 0):.4f})\n"
                )
            parts.append("\n")
        
        # 3. 灵感程序
        if inspirations:
            parts.append("## 灵感程序（供参考的高分方案）\n")
            for i, prog in enumerate(inspirations[:2]):  # 最多 2 个
                s = prog.metrics.get('score', 0.0)
                parts.append(f"### 灵感 {i+1}（适应度: {s:.4f}）\n")
                parts.append(f"```python\n{prog.code}\n```\n")
        
        # 4. 差异格式指令
        parts.append(self.DIFF_INSTRUCTION)
        
        return self.SYSTEM_MESSAGE, '\n'.join(parts)


# 测试
builder = PromptBuilder()
parent = Program(code="def sort(a):\n    return sorted(a)",
                 metrics={"score": 0.5}, generation=3)
sys_msg, user_msg = builder.build_prompt(parent, history=[
    {"description": "尝试用快排", "improved": True, "score": 0.6},
    {"description": "尝试用堆排序", "improved": False, "score": 0.4},
])

print("=== 系统消息 ===")
print(sys_msg[:100], "...")
print("\n=== 用户消息（前 300 字）===")
print(user_msg[:300], "...")

## LLM 集成接口

OpenEvolve 的 LLM 集成有两个层次：

1. **单模型接口** (`llm/openai.py`) — 封装 API 调用、重试、超时
2. **集成接口** (`llm/ensemble.py`) — 管理多个模型，加权随机选择

### 为什么用集成（Ensemble）？

不同的模型有不同的「思维风格」：
- **大模型**（如 GPT-4）：更擅长大幅重构，但慢且贵
- **小模型**（如 GPT-3.5）：更适合小的局部优化，快且便宜

通过加权采样，我们可以让 70% 的变异用小模型（快速迭代），30% 用大模型（大胆创新）。

> **对应源码：** `openevolve/llm/ensemble.py` — `LLMEnsemble` 类

In [ ]:
class LLMInterface:
    """LLM 调用接口的基类。
    
    对应 OpenEvolve: llm/base.py:LLMInterface
    """
    
    def generate(self, system_msg: str, user_msg: str) -> str:
        """生成 LLM 响应。子类必须实现。"""
        raise NotImplementedError


class MockLLM(LLMInterface):
    """模拟 LLM，用于无需 API key 的本地测试。
    
    根据输入代码的模式，生成合理的 SEARCH/REPLACE 修改。
    这让我们可以演示整个流程而不需要真实的 LLM。
    """
    
    def __init__(self, name: str = "mock-llm"):
        self.name = name
        self._strategies = [
            self._optimize_loop,
            self._add_early_exit,
            self._swap_algorithm,
        ]
    
    def generate(self, system_msg: str, user_msg: str) -> str:
        """根据输入选择一种优化策略。"""
        # 从用户消息中提取代码
        code_match = re.search(r'```python\n(.*?)\n```', user_msg, re.DOTALL)
        if not code_match:
            return "无法提取代码"
        code = code_match.group(1)
        
        # 随机选择一种策略
        strategy = random.choice(self._strategies)
        return strategy(code)
    
    def _optimize_loop(self, code: str) -> str:
        """优化循环边界。"""
        lines = code.splitlines()
        for i, line in enumerate(lines):
            if 'range(len(' in line and '- 1)' in line and '- 1 -' not in line:
                old = line.strip()
                new = old.replace('- 1)', '- 1 - i)')
                return f"优化内层循环边界：\n\n<<<<<<< SEARCH\n{old}\n=======\n{new}\n>>>>>>> REPLACE"
        return "未找到可优化的循环"
    
    def _add_early_exit(self, code: str) -> str:
        """添加提前退出条件。"""
        if 'swapped' not in code and 'for i in range' in code:
            return '''添加提前退出标志：

<<<<<<< SEARCH
    for i in range(len(arr)):
=======
    for i in range(len(arr)):
        swapped = False
>>>>>>> REPLACE

<<<<<<< SEARCH
                arr[j], arr[j+1] = arr[j+1], arr[j]
=======
                arr[j], arr[j+1] = arr[j+1], arr[j]
                swapped = True
        if not swapped:
            break
>>>>>>> REPLACE'''
        return "已有提前退出机制"
    
    def _swap_algorithm(self, code: str) -> str:
        """尝试替换为更高效的算法。"""
        if 'bubble' in code.lower() or ('arr[j]' in code and 'arr[j+1]' in code):
            return '''替换为插入排序（通常对小数组更快）：

<<<<<<< SEARCH
    for i in range(len(arr)):
        for j in range(len(arr) - 1):
            if arr[j] > arr[j+1]:
                arr[j], arr[j+1] = arr[j+1], arr[j]
=======
    for i in range(1, len(arr)):
        key = arr[i]
        j = i - 1
        while j >= 0 and arr[j] > key:
            arr[j + 1] = arr[j]
            j -= 1
        arr[j + 1] = key
>>>>>>> REPLACE'''
        return "未检测到可替换的算法模式"


print("MockLLM 就绪 — 3 种内置变异策略")

In [ ]:
class LLMEnsemble:
    """LLM 集成：加权随机选择模型。
    
    对应 OpenEvolve: llm/ensemble.py:LLMEnsemble
    
    每次调用时按权重随机选择一个模型，这样：
    - 大部分变异用快/便宜的模型（高权重）
    - 偶尔用强/贵的模型（低权重）
    - 增加变异的多样性
    """
    
    def __init__(self, models: list[tuple[LLMInterface, float]]):
        """
        Args:
            models: [(model, weight), ...] 列表
        """
        self.models = [m for m, w in models]
        total = sum(w for _, w in models)
        self.weights = [w / total for _, w in models]  # 归一化
    
    def generate(self, system_msg: str, user_msg: str) -> str:
        """加权随机选择模型并生成。"""
        model = random.choices(self.models, weights=self.weights, k=1)[0]
        return model.generate(system_msg, user_msg)
    
    def generate_multiple(self, system_msg: str, user_msg: str, n: int) -> list[str]:
        """生成多个响应（可并行）。
        
        OpenEvolve 用 asyncio 并行调用多个模型。
        我们简化为串行。
        """
        return [self.generate(system_msg, user_msg) for _ in range(n)]


# 创建一个包含 3 个不同策略倾向的 MockLLM 的集成
ensemble = LLMEnsemble([
    (MockLLM("conservative"), 0.5),   # 保守策略（高权重）
    (MockLLM("moderate"), 0.3),       # 中等策略
    (MockLLM("aggressive"), 0.2),     # 激进策略（低权重）
])

print(f"LLM 集成就绪：{len(ensemble.models)} 个模型")
print(f"权重: {ensemble.weights}")

## 集成：智能变异器

把 Prompt 构建器、LLM 集成、Diff 解析器组合成一个完整的智能变异器：

In [ ]:
class SmartMutator:
    """LLM 驱动的智能变异器。
    
    对应 OpenEvolve: iteration.py 中的变异流程
    
    流程：
    1. 构建 prompt
    2. 调用 LLM 集成
    3. 解析 SEARCH/REPLACE 差异
    4. 应用到父代代码
    """
    
    def __init__(self, llm: LLMEnsemble, prompt_builder: PromptBuilder = None):
        self.llm = llm
        self.prompt_builder = prompt_builder or PromptBuilder()
        self.history: list[dict] = []  # 进化历史
    
    def mutate(self, parent: Program,
              inspirations: list[Program] = None) -> Optional[str]:
        """对父代程序进行智能变异。
        
        Returns:
            变异后的代码字符串，如果变异失败则返回 None
        """
        # 1. 构建 prompt
        sys_msg, user_msg = self.prompt_builder.build_prompt(
            parent, self.history, inspirations
        )
        
        # 2. 调用 LLM
        response = self.llm.generate(sys_msg, user_msg)
        
        # 3. 提取差异
        diffs = extract_diffs(response)
        if not diffs:
            return None  # LLM 没有给出有效的修改
        
        # 4. 应用差异
        child_code = apply_diff(parent.code, response)
        
        # 如果代码没有变化，视为失败
        if child_code == parent.code:
            return None
        
        return child_code
    
    def record_attempt(self, description: str, score: float, improved: bool):
        """记录一次尝试到进化历史。"""
        self.history.append({
            "description": description,
            "score": score,
            "improved": improved,
        })
        # 只保留最近 10 条
        if len(self.history) > 10:
            self.history = self.history[-10:]


# 测试
mutator = SmartMutator(ensemble)

parent = Program(
    code="""def sort(arr):
    for i in range(len(arr)):
        for j in range(len(arr) - 1):
            if arr[j] > arr[j+1]:
                arr[j], arr[j+1] = arr[j+1], arr[j]
    return arr""",
    metrics={"score": 0.4},
    generation=5,
)

# 尝试 5 次变异
print("对父代程序尝试 5 次 LLM 变异：\n")
for attempt in range(5):
    child_code = mutator.mutate(parent)
    if child_code:
        # 显示差异
        changed_lines = []
        for old, new in zip(parent.code.splitlines(), child_code.splitlines()):
            if old != new:
                changed_lines.append(f"  - {old.strip()} -> {new.strip()}")
        if child_code.count('\n') != parent.code.count('\n'):
            changed_lines.append(f"  (行数变化: {parent.code.count(chr(10))+1} -> {child_code.count(chr(10))+1})")
        print(f"尝试 {attempt+1}: 成功")
        for cl in changed_lines[:3]:
            print(cl)
    else:
        print(f"尝试 {attempt+1}: 未产生有效变异")
    print()

## 完整演示：LLM 驱动的进化循环

让我们将智能变异器集成到完整的岛屿进化系统中，对比随机变异和 LLM 变异的效果：

In [ ]:
from our_implementation.core import evaluate_sorting, mutate_random, INITIAL_PROGRAM
from our_implementation.islands import IslandSystem

def run_smart_evolution(
    initial_code: str,
    evaluator,
    num_islands: int = 3,
    iterations: int = 100,
) -> tuple:
    """运行 LLM 驱动的多岛屿进化。"""
    system = IslandSystem(
        num_islands=num_islands,
        feature_dims=["complexity"],
        n_bins=5,
        migration_interval=20,
    )
    
    smart_mut = SmartMutator(ensemble)
    
    # 播种
    for i in range(num_islands):
        p = Program(code=initial_code, metrics=evaluator(initial_code), generation=0)
        system.add(p, island_id=i)
    
    best_history = []
    
    for t in range(iterations):
        island_id = t % num_islands
        parent = system.sample(island_id=island_id)
        
        # 用智能变异器
        child_code = smart_mut.mutate(parent)
        
        if child_code is None:
            # LLM 变异失败，回退到随机变异
            child_code = mutate_random(parent.code)
        
        child_metrics = evaluator(child_code)
        child = Program(code=child_code, metrics=child_metrics,
                        generation=parent.generation + 1)
        
        system.add(child, island_id=island_id)
        system.increment_generation(island_id)
        
        # 记录到进化历史
        improved = child_metrics.get("score", 0) > parent.metrics.get("score", 0)
        smart_mut.record_attempt(
            description=f"iteration {t}",
            score=child_metrics.get("score", 0),
            improved=improved,
        )
        
        if system.should_migrate():
            system.migrate()
        
        best_score = system.global_best.metrics.get("score", 0) if system.global_best else 0
        best_history.append(best_score)
    
    return system, best_history

random.seed(42)
print("运行 LLM 驱动的进化...")
smart_system, smart_history = run_smart_evolution(
    INITIAL_PROGRAM, evaluate_sorting, iterations=100
)
print(f"最终最佳分数: {smart_history[-1]:.4f}")

In [ ]:
import matplotlib.pyplot as plt

# 对比：随机变异 vs LLM 变异
random.seed(42)

# 随机变异基线
from our_implementation.map_elites import MAPElitesGrid
baseline_grid = MAPElitesGrid(feature_dims=["complexity"], n_bins=5)
p0 = Program(code=INITIAL_PROGRAM, metrics=evaluate_sorting(INITIAL_PROGRAM), generation=0)
baseline_grid.add(p0)
baseline_history = []

for t in range(100):
    parent = baseline_grid.sample()
    child_code = mutate_random(parent.code)
    child = Program(code=child_code, metrics=evaluate_sorting(child_code),
                    generation=parent.generation + 1)
    baseline_grid.add(child)
    baseline_history.append(
        baseline_grid.best.metrics.get("score", 0) if baseline_grid.best else 0
    )

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(baseline_history, label="随机变异", color="#6b7280", linewidth=2)
ax.plot(smart_history, label="LLM 变异 (MockLLM)", color="#2563eb", linewidth=2)
ax.set_xlabel("迭代次数")
ax.set_ylabel("最佳分数")
ax.set_title("随机变异 vs. LLM 智能变异")
ax.legend()
plt.tight_layout()
plt.show()

print(f"随机变异最终: {baseline_history[-1]:.4f}")
print(f"LLM 变异最终: {smart_history[-1]:.4f}")
print("\n注意：这里用的是 MockLLM，真实 LLM 的效果会更好。")
print("MockLLM 的优势在于它的变异是结构化的（优化循环、添加早退、切换算法），")
print("而随机变异大多数时候只是在破坏代码。")

## 使用真实 LLM

上面我们用 `MockLLM` 演示了整个流程。要使用真实的 LLM，只需替换接口：

```python
import openai

class OpenAILLM(LLMInterface):
    def __init__(self, model="gpt-4", api_key=None):
        self.client = openai.OpenAI(api_key=api_key)
        self.model = model
    
    def generate(self, system_msg, user_msg):
        response = self.client.chat.completions.create(
            model=self.model,
            messages=[
                {"role": "system", "content": system_msg},
                {"role": "user", "content": user_msg},
            ],
            temperature=0.8,  # 较高温度增加变异多样性
        )
        return response.choices[0].message.content
```

OpenEvolve 在 `llm/openai.py` 中实现了完整版本，包括：
- 重试逻辑（默认 3 次，指数退避）
- 超时保护（默认 60 秒）
- 推理模型的特殊处理（不同的参数格式）
- 手动模式（人在回路：人类替代 LLM 输入修改）

## 对应 OpenEvolve 源码

| 我们的实现 | OpenEvolve 源码 | 说明 |
|-----------|----------------|------|
| `extract_diffs()` | `utils/code_utils.py:extract_diffs()` | 正则解析一致 |
| `apply_diff()` | `utils/code_utils.py:apply_diff()` | OpenEvolve 还支持双目标模式 |
| `PromptBuilder` | `prompt/sampler.py:PromptSampler` | OpenEvolve 有模板系统+片段系统 |
| `LLMInterface` | `llm/base.py:LLMInterface` | 抽象基类一致 |
| `LLMEnsemble` | `llm/ensemble.py:LLMEnsemble` | OpenEvolve 支持异步并行生成 |
| `SmartMutator` | `iteration.py:run_iteration()` | 单次迭代的完整变异流程 |
| 进化历史 | `prompt/sampler.py` 中的 previous_attempts | OpenEvolve 还记录差异摘要 |

## 保存到 `our-implementation/`

In [ ]:
MODULE_CODE = '''
"""our-implementation/llm_mutator.py — LLM 驱动的智能变异系统。

在第四章中构建。实现了 Prompt 构建、Diff 解析/应用、LLM 集成。
"""
import re
import random
from typing import Optional, List, Tuple
from .core import Program


def extract_diffs(text: str) -> list[tuple[str, str]]:
    """从 LLM 响应中提取 SEARCH/REPLACE 块。"""
    pattern = r"<<<<<<< SEARCH\\n(.*?)=======\\n(.*?)>>>>>>> REPLACE"
    matches = re.findall(pattern, text, re.DOTALL)
    return [(s.strip("\\n"), r.strip("\\n")) for s, r in matches]


def apply_diff(original_code: str, diff_text: str) -> str:
    """将 SEARCH/REPLACE 差异应用到原始代码。"""
    diffs = extract_diffs(diff_text)
    result = original_code
    for search, replace in diffs:
        if search in result:
            result = result.replace(search, replace, 1)
        else:
            search_lines = search.strip().splitlines()
            result_lines = result.splitlines()
            for i in range(len(result_lines) - len(search_lines) + 1):
                window = result_lines[i:i + len(search_lines)]
                if all(a.strip() == b.strip() for a, b in zip(window, search_lines)):
                    result_lines[i:i + len(search_lines)] = replace.splitlines()
                    result = "\\n".join(result_lines)
                    break
    return result


class LLMInterface:
    """LLM 调用接口基类。"""
    def generate(self, system_msg: str, user_msg: str) -> str:
        raise NotImplementedError


class LLMEnsemble:
    """LLM 集成：加权随机选择模型。"""
    def __init__(self, models: list[tuple]):
        self.models = [m for m, w in models]
        total = sum(w for _, w in models)
        self.weights = [w / total for _, w in models]

    def generate(self, system_msg: str, user_msg: str) -> str:
        model = random.choices(self.models, weights=self.weights, k=1)[0]
        return model.generate(system_msg, user_msg)


class PromptBuilder:
    """构建 LLM 变异提示词。"""
    SYSTEM_MESSAGE = "You are an expert software engineer improving code."

    def build_prompt(self, parent, history=None, inspirations=None):
        parts = [f"## Current program (score: {parent.metrics.get(\"score\", 0):.4f})\\n"]
        parts.append(f"```python\\n{parent.code}\\n```\\n")
        if history:
            parts.append("## Recent attempts\\n")
            for h in history[-3:]:
                parts.append(f"- {h.get(\"description\", \"?\")} -> {h.get(\"score\", 0):.4f}\\n")
        if inspirations:
            parts.append("## Inspirations\\n")
            for p in inspirations[:2]:
                parts.append(f"```python\\n{p.code}\\n```\\n")
        return self.SYSTEM_MESSAGE, "\\n".join(parts)


class SmartMutator:
    """LLM 驱动的智能变异器。"""
    def __init__(self, llm, prompt_builder=None):
        self.llm = llm
        self.prompt_builder = prompt_builder or PromptBuilder()
        self.history = []

    def mutate(self, parent, inspirations=None):
        sys_msg, user_msg = self.prompt_builder.build_prompt(
            parent, self.history, inspirations)
        response = self.llm.generate(sys_msg, user_msg)
        diffs = extract_diffs(response)
        if not diffs:
            return None
        child_code = apply_diff(parent.code, response)
        return child_code if child_code != parent.code else None

    def record_attempt(self, description, score, improved):
        self.history.append({"description": description, "score": score, "improved": improved})
        if len(self.history) > 10:
            self.history = self.history[-10:]
'''

with open("../our-implementation/llm_mutator.py", "w", encoding="utf-8") as f:
    f.write(MODULE_CODE)

print("已保存 our-implementation/llm_mutator.py")
print("  导出: extract_diffs, apply_diff, LLMInterface, LLMEnsemble,")
print("         PromptBuilder, SmartMutator")

## 本章小结

我们构建了：

- **`extract_diffs()`** — 从 LLM 响应中提取 SEARCH/REPLACE 块
- **`apply_diff()`** — 将差异应用到源代码（逐行匹配）
- **`PromptBuilder`** — 组装包含代码、历史、灵感的提示词
- **`LLMEnsemble`** — 加权随机选择多个 LLM 模型
- **`SmartMutator`** — 端到端的智能变异流程

### 核心要点

1. **LLM 理解代码语义** — 变异从随机破坏变成有意义的改进
2. **SEARCH/REPLACE 格式** — 比完整重写更省 token、更精确、更可审计
3. **进化历史避免重复** — LLM 不会反复提出相同的失败修改
4. **集成增加多样性** — 不同模型带来不同风格的变异
5. **灵感程序跨域启发** — 其他岛屿的优秀方案提供新思路

### 还缺什么？

现在我们的评估器只运行一次。但如果评估很慢（比如训练一个模型），我们不想在明显失败的程序上浪费时间。我们需要**级联评估**——先做快速筛选，通过后再做深度评估。

**下一章：** [第五章 — 级联评估](./05-cascade-evaluation.ipynb)  
我们将实现三阶段过滤，用 10% 的计算成本淘汰 90% 的差程序。